# Understand any stock’s risk (AOM quickstart)

**Goal:** see **what the numbers mean**, not giant raw tables.

**Run cells top to bottom:**  
1. Install `riskmodels-py` (≥0.3.2 for AOM `rm` / `run`).  
2. Set **`RISKMODELS_API_KEY`** (Colab Secrets or paste). Get a key at [riskmodels.app/get-key](https://riskmodels.app/get-key).  
3. Four demos: TSLA attribution, AAPL vs NVDA comparison, NVDA exposure snapshot, exposure→hedge chain.

**Charts:** **§1** plots cumulative sleeves **if** the API returns CFR columns; **§3** plots NVDA variance shares after **`GET /metrics`**. **§2** and **§4** are tables/text only (no figures).

Import package name: **`riskmodels`** (PyPI: **`riskmodels-py`**).

In [ ]:
# riskmodels-py 0.3.2+ includes AOM (`rm`/`run`). Bleeding edge: git+https://github.com/BlueWaterCorp/RiskModels_API.git@main#subdirectory=sdk
%pip install -q "riskmodels-py>=0.3.2,<0.4"

## API key

**Google Colab:** click the **key icon** → **Secrets** → add **`RISKMODELS_API_KEY`** (same name as the env var).  
**Or:** leave Secrets empty and paste when the next cell prompts (input is hidden where supported).

**Local Jupyter:** export `RISKMODELS_API_KEY` before launching, or paste in the next cell.

In [ ]:
import os


def ensure_riskmodels_api_key() -> None:
    existing = (os.environ.get("RISKMODELS_API_KEY") or "").strip()
    if existing:
        print("Using RISKMODELS_API_KEY from the environment.")
        return
    try:
        from google.colab import userdata

        secret = userdata.get("RISKMODELS_API_KEY")
        if secret:
            os.environ["RISKMODELS_API_KEY"] = str(secret).strip()
            print("Loaded RISKMODELS_API_KEY from Colab Secrets.")
            return
    except ImportError:
        pass
    except Exception as e:
        print("Colab Secrets not available:", e)
    from getpass import getpass

    os.environ["RISKMODELS_API_KEY"] = getpass("Paste RISKMODELS_API_KEY (hidden where supported): ").strip()
    if not os.environ["RISKMODELS_API_KEY"]:
        raise RuntimeError("RISKMODELS_API_KEY is required. Get a key at https://riskmodels.app/get-key")


ensure_riskmodels_api_key()

## 1. TSLA — “why did it move?” (return attribution)

`.explain()` records **intent** for agents and downstream tools; **`run()`** still returns structured steps today—we spell out the takeaway in prints below. **One chart** below: cumulative sleeves (nothing else—keep cognitive load low).

In [ ]:
%matplotlib inline

from typing import Any

import pandas as pd

from riskmodels import RiskModelsClient, rm, run
from riskmodels.aom import stock


# Long reads from Colab (GET /metrics) often exceed httpx default 120s.
import os

os.environ.setdefault("RISKMODELS_HTTP_TIMEOUT", "300")

client = RiskModelsClient.from_env()

req = (
    rm()
    .subject(stock("TSLA"))
    .scope(date_range_preset="ytd", as_of="latest")
    .return_attribution(resolution="full_stack", view="timeseries")
    .explain()
)

out = run(client, req)


def _first_ticker_returns_df(steps: list[dict[str, Any]]) -> pd.DataFrame | None:
    for step in steps:
        if step.get("op") != "rest_fetch" or step.get("client_method") != "get_ticker_returns":
            continue
        res = step.get("result")
        if isinstance(res, pd.DataFrame) and not res.empty:
            return res
    return None


df = _first_ticker_returns_df(out["steps_out"])
if df is None:
    print("No ticker-returns DataFrame in steps_out (see errors):", out.get("errors"))
    print("(§1 cumulative chart needs a successful get_ticker_returns row set.)\n")
elif out.get("errors"):
    print("Warnings/errors:", out["errors"])
else:
    er_cols = [c for c in ("l3_market_er", "l3_sector_er", "l3_subsector_er", "l3_residual_er") if c in df.columns]
    print("Recent dates — L3 explained-risk variance shares (sum ~100%):")
    print(df[er_cols].tail(5).to_string())
    last = df.iloc[-1]
    print()
    print("Key takeaway (latest row):")
    if er_cols:
        labels = ("Market", "Sector", "Subsector", "Residual (idiosyncratic)")
        pct = [float(last[c]) * 100.0 for c in er_cols]
        for lb, p in zip(labels, pct):
            print(f"  {lb}: {p:.1f}%")
        i_max = max(range(len(pct)), key=lambda j: pct[j])
        print()
        print(f"Largest sleeve right now: {labels[i_max]} ({pct[i_max]:.1f}% of variance).")

# Chart — what drove returns (cumulative sleeves): L1/L2/L3 CFR increments + residual (wire names differ from l3_market_return).
import matplotlib.pyplot as plt

_has_cfr = (
    df is not None
    and not df.empty
    and all(c in df.columns for c in ("date", "l1_combined_factor_return", "l3_residual_return"))
)
if _has_cfr:
    _ts = df.sort_values("date")
    _dt = pd.to_datetime(_ts["date"])
    _n = lambda s: pd.to_numeric(s, errors="coerce").fillna(0.0)
    _mkt = _n(_ts["l1_combined_factor_return"])
    _l2 = _n(_ts["l2_combined_factor_return"]) if "l2_combined_factor_return" in _ts.columns else _mkt
    _l3 = _n(_ts["l3_combined_factor_return"]) if "l3_combined_factor_return" in _ts.columns else _l2
    _sleeves = pd.DataFrame(
        {
            "Market": _mkt,
            "Sector": _l2 - _mkt,
            "Subsector": _l3 - _l2,
            "Residual": _n(_ts["l3_residual_return"]),
        },
        index=_dt,
    )
    _, ax = plt.subplots(figsize=(9, 4))
    _sleeves.cumsum().plot.area(ax=ax, stacked=True, alpha=0.85)
    ax.set_title("TSLA — cumulative return by sleeve (same window as above)")
    ax.set_ylabel("Cumulative return")
    ax.set_xlabel("Date")
    ax.legend(loc="upper left", fontsize=8)
    plt.tight_layout()
    plt.show()
elif df is not None and not df.empty:
    print("(Skipping attribution chart: no L1 CFR / residual columns — unsupported asset type.)")


## 2. Comparison — AAPL vs NVDA

In [ ]:
import pandas as pd

from riskmodels.aom import comparison

req = (
    rm()
    .subject(comparison([stock("AAPL"), stock("NVDA")]))
    .scope(date_range_preset="ytd")
    .return_attribution(resolution="full_stack", view="timeseries")
    .structured()
)

out = run(client, req)

align = next((s for s in out["steps_out"] if s.get("op") == "align_comparison"), None)
if align is None:
    print(out)
else:
    legs = align["result"]["leg_results"]
    tickers = ["AAPL", "NVDA"]
    er_cols = [c for c in ("l3_market_er", "l3_sector_er", "l3_subsector_er", "l3_residual_er")]
    for i, leg_df in enumerate(legs):
        if not isinstance(leg_df, pd.DataFrame):
            continue
        name = tickers[i] if i < len(tickers) else f"leg_{i}"
        cols = [c for c in er_cols if c in leg_df.columns]
        print(f"\n{name} — latest L3 variance shares (tail):")
        print(leg_df[cols].tail(3).to_string())

## 3. Exposure snapshot

**Second (and last) chart** in this notebook: variance shares by sleeve—answers “what do I actually own?”

**Latency:** Section 3 runs **`GET /metrics`** once (`NVDA`). From Google Colab, the **first** snapshot request often takes **~30s–3 minutes** (egress + API)—normal; the cell is not frozen. The code cell prints elapsed seconds when done.
**Timeouts:** If you see **`read operation timed out`**, the HTTP wait exceeded **`RISKMODELS_HTTP_TIMEOUT`** (seconds). The default below is **300**. Restart runtime after changing, or set **`os.environ["RISKMODELS_HTTP_TIMEOUT"] = "600"`** before **`RiskModelsClient.from_env()`**.


In [ ]:
import time

import pandas as pd

%matplotlib inline

req = (
    rm()
    .subject(stock("NVDA"))
    .scope(as_of="latest")
    .exposure(resolution="full_stack", view="snapshot")
    .structured()
)

print("Calling GET /metrics (NVDA snapshot) — may take 30s–3min from Colab...")
_t0 = time.perf_counter()
out = run(client, req)
print(f"API round-trip finished in {time.perf_counter() - _t0:.1f}s")

fetch = next((s for s in out["steps_out"] if s.get("client_method") == "get_metrics"), None)
if fetch is None:
    print(out)
elif fetch.get("error"):
    print("get_metrics failed:", fetch.get("error"))
    print(out)
else:
    res = fetch["result"]
    if isinstance(res, pd.DataFrame):
        row = res.iloc[0]
    elif isinstance(res, dict):
        row = pd.Series(res)
    else:
        raise TypeError(f"Unexpected get_metrics result type: {type(res)}")

    print("NVDA — snapshot (explained risk at L3 = variance shares)")
    for title, col in (
        ("Market", "l3_market_er"),
        ("Sector", "l3_sector_er"),
        ("Subsector", "l3_subsector_er"),
        ("Residual", "l3_residual_er"),
    ):
        if col in row.index and pd.notna(row[col]):
            print(f"  {title}: {float(row[col]) * 100:.1f}%")

    import matplotlib.pyplot as plt

    _labels = ["Market", "Sector", "Subsector", "Residual"]
    _keys = ["l3_market_er", "l3_sector_er", "l3_subsector_er", "l3_residual_er"]
    _vals = [
        float(row[k]) * 100.0 if k in row.index and pd.notna(row[k]) else 0.0 for k in _keys
    ]
    plt.figure(figsize=(6, 3.5))
    plt.bar(_labels, _vals, color=["#1f4e79", "#2e86ab", "#6c757d", "#28a745"])
    plt.title('NVDA — what you "own" by sleeve (variance share)')
    plt.ylabel("% of variance")
    plt.tight_layout()
    plt.show()

    print("Hedge ratios (HR) from snapshot — compare to actionable ETF hedges in the chain demo below:")
    for title, col in (
        ("Market HR", "l3_market_hr"),
        ("Sector HR", "l3_sector_hr"),
        ("Subsector HR", "l3_subsector_hr"),
    ):
        if col in row.index and pd.notna(row[col]):
            print(f"  {title}: {float(row[col]):.3f}")


## 4. Exposure → hedge chain

In [ ]:
from riskmodels.aom import analyze, hedge_action

req = (
    rm()
    .subject(stock("NVDA"))
    .scope(as_of="latest")
    .chain(
        analyze(lens="exposure", resolution="full_stack", view="snapshot"),
        hedge_action(),
    )
    .structured()
)

out = run(client, req)

hedge_step = next((s for s in reversed(out["steps_out"]) if s.get("op") == "hedge_action"), None)
if hedge_step is None:
    print(out)
else:
    body = hedge_step["result"]
    if not isinstance(body, dict):
        print(body)
    else:
        exp = body.get("exposure") or {}
        print("Actionable hedges (POST /decompose) — ETF per layer, HR ≈ hedge ratio convention in API docs:")
        for layer in ("market", "sector", "subsector", "residual"):
            d = exp.get(layer) or {}
            etf = d.get("hedge_etf")
            if etf is None:
                continue
            hr = d.get("hr")
            er = d.get("er")
            hr_s = f"{float(hr):.4f}" if hr is not None else "n/a"
            er_s = f"{float(er) * 100:.1f}%" if er is not None else "n/a"
            print(f"  {layer:10}  {str(etf):6}  HR {hr_s:>10}  variance share {er_s}")